# Tutorial: MAGMA Gene-Set and Gene Property Analysis

This tutorial demonstrates the MAGMA functions added to `cellink` for cell-type enrichment
analysis using GWAS summary statistics:

1. **MAGMA gene-set analysis (GSA)**: test whether top-scoring genes per cell type are enriched for GWAS signal.
2. **MAGMA gene property analysis (GPA)**: test the linear relationship between continuous per-gene scores and GWAS gene-level z-scores.

All functions are available at `cellink.tl.external`. MAGMA must be installed separately; download it from [https://ctg.cncr.nl/software/magma](https://ctg.cncr.nl/software/magma).

### What is a cell-type specificity score?

The central input to both analyses is a **specificity score**: for each gene and each cell type, a number capturing how selectively that gene is expressed in that cell type relative to all others. A gene with high specificity for, say, CD8 Naive T cells is expressed much more in that cell type than anywhere else; a gene with low specificity is expressed broadly across many cell types and isn't very informative about CD8 Naive biology specifically.

This tutorial computes specificity using the method from [Duncan et al. 2025](https://www.nature.com/articles/s41593-024-01834-w): for each gene, take its total expression across all cell types and compute what *fraction* of that total comes from each individual cell type. A gene's specificity scores across cell types therefore sum to 1: a gene expressed equally everywhere has low specificity everywhere, while a gene expressed almost exclusively in one cell type has a specificity score near 1 there and near 0 elsewhere. The working assumption is that genes highly specific to a cell type are more likely to be the genes through which that cell type's biology contributes to disease risk, so testing whether GWAS signal concentrates near these genes is how we link a cell type to a trait.

> **Where this connects to LDSC:** these same specificity scores were originally developed for stratified LD-score regression (S-LDSC), a separate method for partitioning trait heritability across annotations using GWAS summary statistics and a population reference panel. `cellink`'s `preprocess_for_sldsc` function computes the scores (hence the name), but the scores themselves (and everything in this notebook) are used independently of LDSC; MAGMA's gene-set and gene-property tests consume them directly. If you want the full S-LDSC heritability-partitioning workflow instead, see the companion notebook `cell_level_ldsc_analysis.ipynb`.

### GSA vs GPA: which one should I use?

Both tests start from the same specificity scores but ask the question differently:

- **GSA** thresholds each cell type's scores to a top-N% gene set (binary: in the set or not) and tests whether that set is enriched for GWAS signal. Simple to interpret, mirrors a standard binary enrichment test, and a good default, especially if you want results comparable to a published top-10%-style analysis.
- **GPA** uses the full continuous score for every gene (no thresholding) and tests the linear relationship between score and GWAS gene-level association. It avoids the arbitrary top-N% cutoff and can be more powerful when specificity varies smoothly across genes rather than splitting cleanly into "specific" vs "not."

If you're unsure, start with GSA for its simplicity; switch to GPA if a hard top-N% cutoff feels like it's throwing away real signal in your data.

### Required inputs

| Input | Description |
|---|---|
| `specificity_df` | Specific genes × cell-types DataFrame of per-gene scores (e.g. from `preprocess_for_sldsc`) |
| Gene coordinate file | Tab-separated file mapping genes to chr/start/end positions |
| GWAS SNP location file (for MAGMA Step I) | SNP/CHR/BP columns |
| MAGMA gene location file | Fetched from Ensembl via `get_magma_gene_loc` |
| PLINK bfile (for MAGMA Step II) | LD reference panel, e.g. 1000 Genomes EUR |
| GWAS p-value file (for MAGMA Step II) | SNP + P columns |
| `.genes.raw` file (for MAGMA Step III) | Output of Step II; can be pre-computed |


## Environment Setup

In [1]:
import io
import os
import platform
import stat
import zipfile

import requests

# Download the MAGMA v1.10 build for this operating system
# Official source / full download table: https://cncr.nl/research/magma/
MAGMA_SHARE_IDS = {"Linux": "lxDgt2dNdNr6DYt", "Darwin": "1M1d9vHtVidEwvU"}
system = platform.system()
if system not in MAGMA_SHARE_IDS:
    raise RuntimeError(f"No MAGMA v1.10 download is set up for {system!r}; get a build from https://cncr.nl/research/magma/")
url = f"https://vu.data.surf.nl/public.php/dav/files/{MAGMA_SHARE_IDS[system]}/?accept=zip"

r = requests.get(url, timeout=600)
r.raise_for_status()

with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    z.extractall("magma_bin")

# Make all extracted files executable
for root, dirs, files in os.walk("magma_bin"):
    for file in files:
        fpath = os.path.join(root, file)
        st = os.stat(fpath)
        os.chmod(fpath, st.st_mode | stat.S_IEXEC)

In [2]:
import numpy as np
import pandas as pd
import os

from cellink._core import DAnn
from cellink.resources import get_dummy_onek1k, get_gwas_catalog_study_summary_stats
from cellink.tl.external import (
    preprocess_for_sldsc,
    generate_sldsc_genesets,
    scores_to_gmt,
    scores_to_covar,
    genesets_dir_to_entrez_gmt,
    run_magma_annotate,
    get_magma_gene_loc,
    run_magma_gene_analysis,
    run_magma_gsa,
    run_magma_gpa,
)
from cellink.resources import get_1000genomes_plink_files

# Path to the downloaded MAGMA binary (extracted by the cell above)
MAGMA_BIN = os.path.join("magma_bin", "magma")

# MAGMA gene location file: fetched from Ensembl and cached in ~/cellink_data/magma/
MAGMA_GENE_LOC = get_magma_gene_loc(genome_build="GRCh37")

# 1000G PLINK files: downloaded and cached in ~/cellink_data/1000genomes_plink_EUR/
_plink_path, _plink_prefix = get_1000genomes_plink_files(
    population="EUR",
)
# Use chr22 as the LD reference for the tutorial (smallest autosome; use all chroms in production)
G1000_BFILE = str(_plink_path / f"{_plink_prefix}22")

# Cell type for single-cell examples
CELL_TYPE = "CD8 Naive"
celltype_key = "predicted.celltype.l2"
original_donor_col = "donor_id"

site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[2026-11-15 19:05:03,265] INFO:cellink.tl.external._sldsc_utils: Using cached MAGMA gene location file: ~/cellink_data/magma/gene_loc_GRCh37.txt
[2026-11-15 19:05:03,355] INFO:root: ~/cellink_data/1000genomes_plink_EUR/1000G_Phase3_plinkfiles.tgz already exists
[2026-11-15 19:05:03,356] WARNING:root: No checksum provided, skipping verification


## Load and Prepare Data

We load the dummy OneK1K dataset (a 300-donor subset of OneK1K) and compute cell-type specificity scores as described above. (If you've already run the companion LDSC notebook, these preprocessing steps are identical; only the downstream analysis differs.)


In [3]:
dd = get_dummy_onek1k()
print(f"Dataset shape: {dd.shape}")
dd.C.obs[DAnn.donor] = dd.C.obs[original_donor_col]

[2026-11-15 19:05:14,132] INFO:root: ~/cellink_data/dummy_onek1k/dummy_onek1k_dataset.dd.h5 already exists
[2026-11-15 19:05:14,133] INFO:root: Verifying checksum
[2026-11-15 19:05:18,839] INFO:root: Loaded dummy OneK1K dataset: (300, 146939, 125913, 34830)
Dataset shape: (300, 146939, 125913, 34830)


In [4]:
# The dummy dataset is already small, but this shows the subsampling pattern
# you'd use on the full OneK1K dataset (~1.25M cells) for per-cell-type
# specificity estimation.
np.random.seed(0)
n_cells_keep = 30_000
cell_idx = np.sort(np.random.choice(dd.shape[2], min(n_cells_keep, dd.shape[2]), replace=False))
dd = dd[:, :, cell_idx, :].copy()
print(f"Subsampled to {dd.shape[2]:,} cells")

Subsampled to 30,000 cells


In [5]:
dd.C.var["gene"] = dd.C.var_names
adata = dd.C.copy()

adata_filtered, mean_expr, specificity = preprocess_for_sldsc(
    adata,
    celltype_col=celltype_key,
    gene_identifier_mode="ensembl",
    gene_col="gene",
    genome_build="GRCh37",
    inplace=False,
)
print(f"Specificity shape: {specificity.shape}")
specificity.head()

[2026-11-15 19:05:19,693] INFO:cellink.tl.external._sldsc_utils: Fetching gene annotations from GRCh37...
[2026-11-15 19:05:19,693] INFO:cellink.tl.external._biomart: Querying Ensembl BioMart at https://grch37.ensembl.org (hsapiens_gene_ensembl: hgnc_symbol, external_gene_name, ensembl_gene_id, chromosome_name, start_position, end_position, gene_biotype)
[2026-11-15 19:05:23,152] INFO:cellink.tl.external._sldsc_utils: Fetched annotations for 63763 genes from GRCh37
[2026-11-15 19:05:23,227] INFO:cellink.tl.external._sldsc_utils: Removing version suffixes from Gene IDs
[2026-11-15 19:05:23,592] INFO:cellink.tl.external._sldsc_utils: Dropping conflicting columns from adata.var before merge: ['chrom', 'start', 'end']
[2026-11-15 19:05:23,993] INFO:cellink.tl.external._sldsc_utils: Annotated 30793 / 34830 genes.
[2026-11-15 19:05:24,000] INFO:cellink.tl.external._sldsc_utils: Using annotation columns: gene=gene, biotype=gene_biotype, chr=chrom, start=start, end=end
[2026-11-15 19:05:24,001

Aggregating clusters: 100%|██████████| 31/31 [00:01<00:00, 23.75it/s]

[2026-11-15 19:05:25,878] INFO:cellink.tl.external._sldsc_utils: Log1p applied.
[2026-11-15 19:05:25,898] INFO:cellink.tl.external._sldsc_utils: Computing mean expression for predicted.celltype.l2


[2026-11-15 19:05:26,064] INFO:cellink.tl.external._sldsc_utils: Computing specificity scores
[2026-11-15 19:05:26,192] INFO:cellink.tl.external._sldsc_utils: Final data shape: (30000, 15074)
[2026-11-15 19:05:26,193] INFO:cellink.tl.external._sldsc_utils: Mean expression shape: (15074, 31)
[2026-11-15 19:05:26,194] INFO:cellink.tl.external._sldsc_utils: Specificity shape: (15074, 31)
Specificity shape: (15074, 31)


ClusterID,ASDC,B intermediate,B memory,B naive,CD14 Mono,CD16 Mono,CD4 CTL,CD4 Naive,CD4 Proliferating,CD4 TCM,...,NK Proliferating,NK_CD56bright,Plasmablast,Platelet,Treg,cDC1,cDC2,dnT,gdT,pDC
gene,,,,,,,,,,,,,,,,,,,,,
ENSG00000000419,0.043900,0.021392,0.024806,0.026755,0.022598,0.038790,0.029550,0.031282,0.069241,0.036090,...,0.032748,0.020236,0.057838,0.018041,0.026935,0.0,0.035158,0.038819,0.033803,0.022770
ENSG00000000457,0.000000,0.041122,0.028927,0.042313,0.030108,0.027554,0.053553,0.048330,0.097751,0.037840,...,0.080930,0.036369,0.033236,0.000000,0.026911,0.0,0.024741,0.036545,0.029008,0.076848
ENSG00000000460,0.000000,0.027218,0.011968,0.011222,0.008304,0.037999,0.006714,0.007928,0.215690,0.013323,...,0.111608,0.016718,0.045834,0.054651,0.005302,0.0,0.011373,0.000000,0.005715,0.029562
ENSG00000000938,0.053209,0.032085,0.020979,0.025174,0.114200,0.135347,0.030727,0.003212,0.004012,0.002192,...,0.039222,0.041872,0.015636,0.037287,0.002761,0.0,0.084332,0.003750,0.042425,0.006599
ENSG00000000971,0.000000,0.000000,0.000000,0.000000,0.017797,0.000000,0.129498,0.001999,0.000000,0.067961,...,0.000000,0.035829,0.000000,0.117123,0.034087,0.0,0.000000,0.000000,0.061238,0.000000


---
## Part 1: MAGMA 

MAGMA provides a complementary enrichment analysis that does not require LD score files. The pipeline has three steps:

| Step | Function | Output |
|---|---|---|
| I: Annotate | `run_magma_annotate` | `.genes.annot` |
| II: Gene analysis | `run_magma_gene_analysis` | `.genes.raw` |
| III-a: Gene-set analysis | `run_magma_gsa` | `.gsa.out` |
| III-b: Gene property analysis | `run_magma_gpa` | `.gsa.out` |

Steps I and II only need to be run once per GWAS. Step III is run once per score set / method.

### Step I: Annotate SNPs to genes

In [6]:
# Derive MAGMA's SNP location file (SNP / CHR / BP) from real GWAS summary
# statistics, the same study (GCST004787, coronary artery disease) used in
# the companion LDSC notebook's heritability/genetic-correlation sections.
gwas_summary_statistic_path = get_gwas_catalog_study_summary_stats(
    "GCST004787", genome_build="GRCh37", return_path=True
)
gwas_df = pd.read_csv(gwas_summary_statistic_path, sep="\t")
gwas_df = gwas_df.dropna(subset=["variant_id", "chromosome", "base_pair_location", "p_value"])
# chromosome/base_pair_location load as float64 (NaN forces the column to
# float dtype), cast to int before filtering/writing, otherwise "22.0" never
# matches "22" and MAGMA receives malformed CHR/BP columns.
gwas_df["chromosome"] = gwas_df["chromosome"].astype(int)
gwas_df["base_pair_location"] = gwas_df["base_pair_location"].astype(int)
gwas_df = gwas_df[gwas_df["chromosome"] == 22]  # restrict to chr22, matching G1000_BFILE
print(f"{len(gwas_df):,} chr22 SNPs with usable GWAS data")

os.makedirs("magma_results", exist_ok=True)
GWAS_SNP_LOC = "magma_results/gwas_snp_loc.txt"
gwas_df[["variant_id", "chromosome", "base_pair_location"]].rename(
    columns={"variant_id": "SNP", "chromosome": "CHR", "base_pair_location": "BP"}
).to_csv(GWAS_SNP_LOC, sep="\t", index=False)

annot_result = run_magma_annotate(
    snp_loc=GWAS_SNP_LOC,
    gene_loc=MAGMA_GENE_LOC,
    out_prefix="magma_results/cad",
    magma_bin=MAGMA_BIN,
    window_kb=0,  # gene body only; use e.g. 35 for ±35 kb
)
print("Annotation file:", annot_result["annot_file"])

[2026-11-15 19:05:26,531] INFO:root: Using cached summary statistics ~/cellink_data/gwas_catalog_summary_stats/GCST004787/28714975-GCST004787-EFO_0001645-Build37.f.tsv.gz (downloaded from http://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/GCST004001-GCST005000/GCST004787/harmonised/28714975-GCST004787-EFO_0001645-Build37.f.tsv.gz, build GRCh37)
121,746 chr22 SNPs with usable GWAS data
[2026-11-15 19:05:35,635] INFO:cellink.tl.external._ldsc2magma: Running: magma_bin/magma --annotate --snp-loc magma_results/gwas_snp_loc.txt --gene-loc ~/cellink_data/magma/gene_loc_GRCh37.txt --out magma_results/cad
[2026-11-15 19:05:35,914] INFO:cellink.tl.external._ldsc2magma: Welcome to MAGMA v1.10 (linux/s)
Using flags:
	--annotate
	--snp-loc magma_results/gwas_snp_loc.txt
	--gene-loc ~/cellink_data/magma/gene_loc_GRCh37.txt
	--out magma_results/cad

Start time is 19:05:35, Sunday 15 Nov 2026

Starting annotation...
Reading gene locations from file ~/cellink_data/magma/gene_loc_GRCh37.txt... 

In [7]:
# Preview the command without running:
cmd_preview = run_magma_annotate(
    snp_loc=GWAS_SNP_LOC,
    gene_loc=MAGMA_GENE_LOC,
    out_prefix="magma_results/cad",
    magma_bin=MAGMA_BIN,
    run=False,
)
print(" ".join(cmd_preview["command"]))

magma_bin/magma --annotate --snp-loc magma_results/gwas_snp_loc.txt --gene-loc ~/cellink_data/magma/gene_loc_GRCh37.txt --out magma_results/cad


### Step II: Gene-level association analysis

This step uses a PLINK LD reference panel to compute gene-level z-scores from the GWAS p-values. The output `.genes.raw` file is the input for both GSA and GPA.

In [8]:
# GWAS p-value file: tab-delimited, columns SNP / P / N. The real summary
# statistics already carry a per-SNP sample size (n_samples), so we pass
# that through directly instead of a single global N.
GWAS_PVAL_FILE = "magma_results/gwas_pval.txt"
gwas_pval_df = gwas_df[["variant_id", "p_value", "n_samples"]].copy()
gwas_pval_df["n_samples"] = gwas_pval_df["n_samples"].astype(int)
gwas_pval_df.rename(columns={"variant_id": "SNP", "p_value": "P", "n_samples": "N"}).to_csv(
    GWAS_PVAL_FILE, sep="\t", index=False
)

gene_result = run_magma_gene_analysis(
    bfile=G1000_BFILE,
    pval_file=GWAS_PVAL_FILE,
    gene_annot=annot_result["annot_file"],
    out_prefix="magma_results/cad",
    ncol="N",  # tells MAGMA to read per-SNP N from the file's "N" column
    magma_bin=MAGMA_BIN,
)
print("Gene results:", gene_result["gene_results"])

[2026-11-15 19:05:36,125] INFO:cellink.tl.external._ldsc2magma: Running: magma_bin/magma --bfile ~/cellink_data/1000genomes_plink_EUR/1000G.EUR.QC.22 --pval magma_results/gwas_pval.txt ncol=N --gene-annot magma_results/cad.genes.annot --out magma_results/cad
[2026-11-15 19:05:55,468] INFO:cellink.tl.external._ldsc2magma: Welcome to MAGMA v1.10 (linux/s)
Using flags:
	--bfile ~/cellink_data/1000genomes_plink_EUR/1000G.EUR.QC.22
	--pval magma_results/gwas_pval.txt ncol=N
	--gene-annot magma_results/cad.genes.annot
	--out magma_results/cad

Start time is 19:05:36, Sunday 15 Nov 2026

Loading PLINK-format data...
Reading file ~/cellink_data/1000genomes_plink_EUR/1000G.EUR.QC.22.fam... 489 individuals read
Reading file ~/cellink_data/1000genomes_plink_EUR/1000G.EUR.QC.22.bim... 141123 SNPs read
Preparing file ~/cellink_data/1000genomes_plink_EUR/1000G.EUR.QC.22.bed... 

Reading SNP p-values from file magma_results/gwas_pval.txt... 
	detected 3 variables in file
	using variable: SNP (SNP id)

In [9]:
# If you already have a pre-computed .genes.raw (e.g. from the combined_pipeline)
# you can skip Steps I and II and point directly to it. We just computed one
# for real above, so we use that:
GENES_RAW = gene_result["gene_results"]

---
## Part 2: MAGMA Gene-Set Analysis (GSA)

GSA tests whether the top-scoring genes in each cell type show stronger GWAS association than background. There are two paths to create the required GMT file:

* **From a scores DataFrame**: use `scores_to_gmt` (selects top-N% per cell type)
* **From existing LDSC `.GeneSet` files**: use `genesets_dir_to_entrez_gmt`

### Path A: From continuous scores

In [10]:
# specificity is a genes × cell-types DataFrame with ENSG IDs as the index
gmt_file = scores_to_gmt(
    scores=specificity,
    out_file="magma_gsa/onek1k_specificity_top10.gmt",
    top_frac=0.10,  # top 10% genes per cell type
    ascending=False,  # ascending=True would select the bottom 10%
    set_name_prefix="onek1k_specificity",  # optional prefix for set names
)
print("GMT file:", gmt_file)

# Preview the first two lines
with open(gmt_file) as f:
    for i, line in enumerate(f):
        parts = line.rstrip().split("\t")
        print(f"Set '{parts[0]}': {len(parts)-2} genes")
        if i == 1:
            break

[2026-11-15 19:05:55,640] INFO:cellink.tl.external._ldsc2magma: scores_to_gmt: wrote 31 gene sets (top 10%, 0 skipped) → magma_gsa/onek1k_specificity_top10.gmt
GMT file: magma_gsa/onek1k_specificity_top10.gmt
Set 'onek1k_specificity_ASDC': 1507 genes
Set 'onek1k_specificity_B_intermediate': 1507 genes


In [11]:
# If gene scores use gene symbols rather than ENSG IDs, provide a mapping:
# gene_map can be a path to a TSV with columns gene_name / ensg_id,
# or a pd.Series indexed by symbol with ENSG values.

# gmt_file = scores_to_gmt(
#     scores=scores_with_symbols,
#     out_file="magma_gsa/scores_mapped.gmt",
#     gene_map="cellink/combined_pipeline/gene_name_to_ensg.tsv",
# )

### Path B: From existing binary LDSC `.GeneSet` files

In [12]:
# First generate the *.GeneSet files (one per cell type: its top 10% of genes by specificity)
generate_sldsc_genesets(specificity, dd.C, out_dir="ldsc_genesets", top_frac=0.10, overwrite=True)

# Converts the *.GeneSet files produced by generate_sldsc_genesets into GMT.
# Gene IDs are passed through as-is (no Ensembl → Entrez conversion).
gmt_from_genesets = genesets_dir_to_entrez_gmt(
    geneset_dir="ldsc_genesets",
    out_gmt="magma_gsa/ldsc_genesets.gmt",
    include_control=False,
)
print("GMT from GeneSet files:", gmt_from_genesets)

[2026-11-15 19:05:55,657] INFO:cellink.tl.external._sldsc_utils: Writing gene sets to ldsc_genesets
[2026-11-15 19:05:55,669] INFO:cellink.tl.external._sldsc_utils: specificity_df index looks like Ensembl IDs; using them directly.
[2026-11-15 19:05:55,671] INFO:cellink.tl.external._sldsc_utils: Selecting top 1508 genes (10.0%) per cell type
[2026-11-15 19:05:56,424] INFO:cellink.tl.external._sldsc_utils: Wrote control gene set with 15074 genes
[2026-11-15 19:05:56,426] INFO:cellink.tl.external._sldsc_utils: Generated 31 cell-type-specific gene sets
[2026-11-15 19:05:56,480] INFO:cellink.tl.external._ldsc2magma: Wrote 31 gene sets to magma_gsa/ldsc_genesets.gmt (skipped 0)
GMT from GeneSet files: magma_gsa/ldsc_genesets.gmt


### Run GSA

In [13]:
gsa_result = run_magma_gsa(
    gene_results=GENES_RAW,
    set_annot=str(gmt_file),
    out_prefix="magma_gsa/cad_onek1k_specificity",
    magma_bin=MAGMA_BIN,
)
print("GSA results:", gsa_result["results_file"])

# Load and display results
gsa_df = pd.read_csv(gsa_result["results_file"], sep=r"\s+", comment="#")
gsa_df.sort_values("P").head(10)

[2026-11-15 19:05:56,487] INFO:cellink.tl.external._ldsc2magma: Running: magma_bin/magma --gene-results magma_results/cad.genes.raw --set-annot magma_gsa/onek1k_specificity_top10.gmt --out magma_gsa/cad_onek1k_specificity
[2026-11-15 19:05:57,479] INFO:cellink.tl.external._ldsc2magma: Welcome to MAGMA v1.10 (linux/s)
Using flags:
	--gene-results magma_results/cad.genes.raw
	--set-annot magma_gsa/onek1k_specificity_top10.gmt
	--out magma_gsa/cad_onek1k_specificity

Start time is 19:05:56, Sunday 15 Nov 2026

Reading file magma_results/cad.genes.raw... 
	943 genes read from file
Loading gene-set annotation...
Reading file magma_gsa/onek1k_specificity_top10.gmt... 
	31 gene-set definitions read from file
	found 31 gene sets containing genes defined in genotype data (containing a total of 363 unique genes)
Preparing variables for analysis...
	truncating Z-scores 3 points below zero or 6 standard deviations above the mean
	truncating covariate values more than 5 standard deviations from the

,VARIABLE,TYPE,NGENES,BETA,BETA_STD,SE,P,FULL_NAME
3,onek1k_specificity_B_naive,SET,32,0.300440,0.054427,0.16295,0.032787,onek1k_specificity_B_naive
11,onek1k_specificity_CD8_Naive,SET,43,0.204950,0.042779,0.12610,0.052236,onek1k_specificity_CD8_Naive
22,onek1k_specificity_NK_CD56br...,SET,44,0.232230,0.049006,0.14521,0.055062,onek1k_specificity_NK_CD56bright
24,onek1k_specificity_Platelet,SET,41,0.200390,0.040887,0.13347,0.066823,onek1k_specificity_Platelet
1,onek1k_specificity_B_interme...,SET,37,0.157220,0.030541,0.13535,0.122860,onek1k_specificity_B_intermediate
12,onek1k_specificity_CD8_Proli...,SET,38,0.116600,0.022941,0.14964,0.218050,onek1k_specificity_CD8_Proliferating
17,onek1k_specificity_HSPC,SET,39,0.098410,0.019605,0.14790,0.253000,onek1k_specificity_HSPC
16,onek1k_specificity_Eryth,SET,39,0.064600,0.012870,0.13361,0.314440,onek1k_specificity_Eryth
27,onek1k_specificity_cDC2,SET,36,0.064749,0.012414,0.14462,0.327230,onek1k_specificity_cDC2
5,onek1k_specificity_CD16_Mono,SET,35,0.052100,0.009855,0.14680,0.361370,onek1k_specificity_CD16_Mono


In [14]:
# Extra MAGMA flags are forwarded via **kwargs, e.g. joint competitive test:
# run_magma_gsa(..., model="multi")

---
## Part 3: MAGMA Gene Property Analysis (GPA)

GPA tests the linear relationship between continuous per-gene scores and GWAS gene z-scores. Unlike GSA it uses all genes with scores: no top-N threshold. The covariate file contains one column per cell type.

### Build the covariate file

In [15]:
covar_file = scores_to_covar(
    scores=specificity,
    out_file="magma_gpa/onek1k_specificity.covar",
    negate=False,  # set negate=True to test enrichment in low-score genes
)
print("Covariate file:", covar_file)

# Preview
pd.read_csv(covar_file, sep="\t", index_col=0).head()

[2026-11-15 19:05:58,044] INFO:cellink.tl.external._ldsc2magma: scores_to_covar: wrote 15074 genes × 31 cell types → magma_gpa/onek1k_specificity.covar
Covariate file: magma_gpa/onek1k_specificity.covar


,ASDC,B_intermediate,B_memory,B_naive,CD14_Mono,CD16_Mono,CD4_CTL,CD4_Naive,CD4_Proliferating,CD4_TCM,...,NK_Proliferating,NK_CD56bright,Plasmablast,Platelet,Treg,cDC1,cDC2,dnT,gdT,pDC
GENE,,,,,,,,,,,,,,,,,,,,,
ENSG00000000419,0.043900,0.021392,0.024806,0.026755,0.022598,0.038790,0.029550,0.031282,0.069241,0.036090,...,0.032748,0.020236,0.057838,0.018041,0.026935,0.0,0.035158,0.038819,0.033803,0.022770
ENSG00000000457,0.000000,0.041122,0.028927,0.042313,0.030108,0.027554,0.053553,0.048330,0.097751,0.037840,...,0.080930,0.036369,0.033236,0.000000,0.026911,0.0,0.024741,0.036545,0.029008,0.076848
ENSG00000000460,0.000000,0.027218,0.011968,0.011222,0.008304,0.037999,0.006714,0.007928,0.215690,0.013323,...,0.111608,0.016718,0.045834,0.054651,0.005302,0.0,0.011373,0.000000,0.005715,0.029562
ENSG00000000938,0.053209,0.032085,0.020979,0.025174,0.114200,0.135347,0.030727,0.003212,0.004012,0.002192,...,0.039222,0.041872,0.015636,0.037287,0.002761,0.0,0.084332,0.003750,0.042425,0.006599
ENSG00000000971,0.000000,0.000000,0.000000,0.000000,0.017797,0.000000,0.129498,0.001999,0.000000,0.067961,...,0.000000,0.035829,0.000000,0.117123,0.034087,0.0,0.000000,0.000000,0.061238,0.000000


In [16]:
# Gene symbols → ENSG mapping works the same way as in scores_to_gmt:
# covar_file = scores_to_covar(
#     scores=scores_with_symbols,
#     out_file="magma_gpa/scores_mapped.covar",
#     gene_map="cellink/combined_pipeline/gene_name_to_ensg.tsv",
# )

### Joint GPA (default)

All cell types are tested in a single MAGMA call. MAGMA may drop highly correlated covariates via its collinearity filter. Use univariate mode if this is a concern.

In [17]:
gpa_result = run_magma_gpa(
    gene_results=GENES_RAW,
    gene_covar=str(covar_file),
    out_prefix="magma_gpa/cad_onek1k_specificity_joint",
    magma_bin=MAGMA_BIN,
    univariate=False,
)
print("GPA results:", gpa_result["results_file"])

gpa_df = pd.read_csv(gpa_result["results_file"], sep=r"\s+", comment="#")
gpa_df.sort_values("P").head(10)

[2026-11-15 19:05:58,091] INFO:cellink.tl.external._ldsc2magma: Running: magma_bin/magma --gene-results magma_results/cad.genes.raw --gene-covar magma_gpa/onek1k_specificity.covar --out magma_gpa/cad_onek1k_specificity_joint
[2026-11-15 19:05:58,262] INFO:cellink.tl.external._ldsc2magma: Welcome to MAGMA v1.10 (linux/s)
Using flags:
	--gene-results magma_results/cad.genes.raw
	--gene-covar magma_gpa/onek1k_specificity.covar
	--out magma_gpa/cad_onek1k_specificity_joint

Start time is 19:05:58, Sunday 15 Nov 2026

Reading file magma_results/cad.genes.raw... 
	943 genes read from file
Loading gene-level covariates...
Reading file magma_gpa/onek1k_specificity.covar... 
	detected 31 variables in file (using all)
	found 31 valid gene covariates, for 390 genes defined in genotype data
Processing missing values...
	found 553 genes not present in all input files: removing these from analysis
	390 genes remaining in analysis
Preparing variables for analysis...
	truncating Z-scores 3 points belo

,VARIABLE,TYPE,NGENES,BETA,BETA_STD,SE,P
3,B_naive,COVAR,390,2.49170,0.092857,1.31190,0.058332
28,dnT,COVAR,390,-1.78030,-0.073668,1.16890,0.128620
25,Treg,COVAR,390,-2.88090,-0.068389,1.91910,0.134210
16,Eryth,COVAR,390,-1.79980,-0.061736,1.26530,0.155780
0,ASDC,COVAR,390,-1.46820,-0.058301,1.03480,0.156850
1,B_intermediate,COVAR,390,1.63860,0.060714,1.24060,0.187400
9,CD4_TCM,COVAR,390,-0.85652,-0.048982,0.80304,0.286870
22,NK_CD56bright,COVAR,390,1.09670,0.051143,1.06330,0.303040
15,Doublet,COVAR,390,-0.65721,-0.040263,0.80300,0.413650
2,B_memory,COVAR,390,-0.71264,-0.034253,0.91036,0.434260


### Univariate GPA

Each cell type is tested in an independent MAGMA call. Slower than joint mode but guarantees a result for every cell type, recommended when covariate columns are highly correlated (e.g. residual CV scores or negated scores).

In [18]:
gpa_univ_result = run_magma_gpa(
    gene_results=GENES_RAW,
    gene_covar=str(covar_file),
    out_prefix="magma_gpa/cad_onek1k_specificity_univariate",
    magma_bin=MAGMA_BIN,
    univariate=True,
)
print("Univariate GPA results:", gpa_univ_result["results_file"])

gpa_univ_df = pd.read_csv(gpa_univ_result["results_file"], sep=r"\s+", comment="#")
gpa_univ_df.sort_values("P").head(10)

[2026-11-15 19:05:58,310] INFO:cellink.tl.external._ldsc2magma: GPA univariate: 31 cell types
[2026-11-15 19:05:58,333] INFO:cellink.tl.external._ldsc2magma: Running: magma_bin/magma --gene-results magma_results/cad.genes.raw --gene-covar /tmp/tmplcac9f8r/ct.covar --out /tmp/tmplcac9f8r/ct_out
[2026-11-15 19:05:58,460] INFO:cellink.tl.external._ldsc2magma: Welcome to MAGMA v1.10 (linux/s)
Using flags:
	--gene-results magma_results/cad.genes.raw
	--gene-covar /tmp/tmplcac9f8r/ct.covar
	--out /tmp/tmplcac9f8r/ct_out

Start time is 19:05:58, Sunday 15 Nov 2026

Reading file magma_results/cad.genes.raw... 
	943 genes read from file
Loading gene-level covariates...
Reading file /tmp/tmplcac9f8r/ct.covar... 
	detected 1 variables in file (using all)
	found 1 valid gene covariate, for 390 genes defined in genotype data
Processing missing values...
	found 553 genes not present in all input files: removing these from analysis
	390 genes remaining in analysis
Preparing variables for analysis...


,VARIABLE,TYPE,NGENES,BETA,BETA_STD,SE,P,FULL_NAME
3,B_naive,COVAR,390,2.49170,0.092857,1.31190,0.058332,B_naive
28,dnT,COVAR,390,-1.78030,-0.073668,1.16890,0.128620,dnT
25,Treg,COVAR,390,-2.88090,-0.068389,1.91910,0.134210,Treg
16,Eryth,COVAR,390,-1.79980,-0.061736,1.26530,0.155780,Eryth
0,ASDC,COVAR,390,-1.46820,-0.058301,1.03480,0.156850,ASDC
1,B_intermediate,COVAR,390,1.63860,0.060714,1.24060,0.187400,B_intermediate
9,CD4_TCM,COVAR,390,-0.85652,-0.048982,0.80304,0.286870,CD4_TCM
22,NK_CD56bright,COVAR,390,1.09670,0.051143,1.06330,0.303040,NK_CD56bright
15,Doublet,COVAR,390,-0.65721,-0.040263,0.80300,0.413650,Doublet
2,B_memory,COVAR,390,-0.71264,-0.034253,0.91036,0.434260,B_memory


---
## Part 4: LDSC → MAGMA: Full Workflows

The two complete workflows from LDSC score outputs to MAGMA results.

### Workflow A: LDSC binary genesets → MAGMA GSA

In [19]:
# 1. Generate LDSC gene sets (top 10% specificity per cell type)
generate_sldsc_genesets(specificity, dd.C, out_dir="ldsc_genesets", top_frac=0.10, overwrite=True)

# 2. Convert .GeneSet → GMT (MAGMA format, ENSG IDs kept as-is)
gmt_ldsc = genesets_dir_to_entrez_gmt(
    geneset_dir="ldsc_genesets",
    out_gmt="magma_gsa/ldsc_genesets_workflow_a.gmt",
)

# 3. GSA
gsa_a = run_magma_gsa(
    gene_results=GENES_RAW,
    set_annot=str(gmt_ldsc),
    out_prefix="magma_gsa/workflow_a_cad",
    magma_bin=MAGMA_BIN,
)

results_a = pd.read_csv(gsa_a["results_file"], sep=r"\s+", comment="#")
print(results_a.sort_values("P").head())

[2026-11-15 19:06:03,460] INFO:cellink.tl.external._sldsc_utils: Writing gene sets to ldsc_genesets
[2026-11-15 19:06:03,467] INFO:cellink.tl.external._sldsc_utils: specificity_df index looks like Ensembl IDs; using them directly.
[2026-11-15 19:06:03,468] INFO:cellink.tl.external._sldsc_utils: Selecting top 1508 genes (10.0%) per cell type
[2026-11-15 19:06:04,034] INFO:cellink.tl.external._sldsc_utils: Wrote control gene set with 15074 genes
[2026-11-15 19:06:04,036] INFO:cellink.tl.external._sldsc_utils: Generated 31 cell-type-specific gene sets
[2026-11-15 19:06:04,108] INFO:cellink.tl.external._ldsc2magma: Wrote 31 gene sets to magma_gsa/ldsc_genesets_workflow_a.gmt (skipped 0)
[2026-11-15 19:06:04,109] INFO:cellink.tl.external._ldsc2magma: Running: magma_bin/magma --gene-results magma_results/cad.genes.raw --set-annot magma_gsa/ldsc_genesets_workflow_a.gmt --out magma_gsa/workflow_a_cad
[2026-11-15 19:06:05,088] INFO:cellink.tl.external._ldsc2magma: Welcome to MAGMA v1.10 (linux/

### Workflow B: Continuous scores → MAGMA GPA

In [20]:
# 1. Build covariate file directly from the specificity score matrix
covar_b = scores_to_covar(
    scores=specificity,
    out_file="magma_gpa/onek1k_specificity_workflow_b.covar",
)

# 2. GPA (joint mode)
gpa_b = run_magma_gpa(
    gene_results=GENES_RAW,
    gene_covar=str(covar_b),
    out_prefix="magma_gpa/workflow_b_cad",
    magma_bin=MAGMA_BIN,
)

results_b = pd.read_csv(gpa_b["results_file"], sep=r"\s+", comment="#")
print(results_b.sort_values("P").head())

[2026-11-15 19:06:05,659] INFO:cellink.tl.external._ldsc2magma: scores_to_covar: wrote 15074 genes × 31 cell types → magma_gpa/onek1k_specificity_workflow_b.covar
[2026-11-15 19:06:05,660] INFO:cellink.tl.external._ldsc2magma: Running: magma_bin/magma --gene-results magma_results/cad.genes.raw --gene-covar magma_gpa/onek1k_specificity_workflow_b.covar --out magma_gpa/workflow_b_cad
[2026-11-15 19:06:05,841] INFO:cellink.tl.external._ldsc2magma: Welcome to MAGMA v1.10 (linux/s)
Using flags:
	--gene-results magma_results/cad.genes.raw
	--gene-covar magma_gpa/onek1k_specificity_workflow_b.covar
	--out magma_gpa/workflow_b_cad

Start time is 19:06:05, Sunday 15 Nov 2026

Reading file magma_results/cad.genes.raw... 
	943 genes read from file
Loading gene-level covariates...
Reading file magma_gpa/onek1k_specificity_workflow_b.covar... 
	detected 31 variables in file (using all)
	found 31 valid gene covariates, for 390 genes defined in genotype data
Processing missing values...
	found 553 ge

---
## Part 5: Tips and Common Patterns

### Previewing commands without running

All MAGMA functions accept `run=False` to return the command list without executing it. Useful for submitting to a cluster.

In [21]:
for fn_name, result in [
    (
        "run_magma_annotate",
        run_magma_annotate(
            snp_loc=GWAS_SNP_LOC,
            gene_loc=MAGMA_GENE_LOC,
            out_prefix="magma_results/cad",
            magma_bin=MAGMA_BIN,
            run=False,
        ),
    ),
    (
        "run_magma_gene_analysis",
        run_magma_gene_analysis(
            bfile=G1000_BFILE,
            pval_file=GWAS_PVAL_FILE,
            gene_annot="magma_results/cad.genes.annot",
            out_prefix="magma_results/cad",
            ncol="N",
            magma_bin=MAGMA_BIN,
            run=False,
        ),
    ),
    (
        "run_magma_gsa",
        run_magma_gsa(
            gene_results=GENES_RAW,
            set_annot=str(gmt_file),
            out_prefix="magma_gsa/preview",
            magma_bin=MAGMA_BIN,
            run=False,
        ),
    ),
    (
        "run_magma_gpa",
        run_magma_gpa(
            gene_results=GENES_RAW,
            gene_covar=str(covar_file),
            out_prefix="magma_gpa/preview",
            magma_bin=MAGMA_BIN,
            run=False,
        ),
    ),
]:
    print(f"{fn_name}:\n  {' '.join(result['command'])}\n")

run_magma_annotate:
  magma_bin/magma --annotate --snp-loc magma_results/gwas_snp_loc.txt --gene-loc ~/cellink_data/magma/gene_loc_GRCh37.txt --out magma_results/cad

run_magma_gene_analysis:
  magma_bin/magma --bfile ~/cellink_data/1000genomes_plink_EUR/1000G.EUR.QC.22 --pval magma_results/gwas_pval.txt ncol=N --gene-annot magma_results/cad.genes.annot --out magma_results/cad

run_magma_gsa:
  magma_bin/magma --gene-results magma_results/cad.genes.raw --set-annot magma_gsa/onek1k_specificity_top10.gmt --out magma_gsa/preview

run_magma_gpa:
  magma_bin/magma --gene-results magma_results/cad.genes.raw --gene-covar magma_gpa/onek1k_specificity.covar --out magma_gpa/preview



### Negating scores to test low-score enrichment

Some metrics (e.g. coefficient of variation) are expected to be *low* in disease-relevant genes. Pass `negate=True` to `scores_to_covar`, or `ascending=True` to `scores_to_gmt`, to select/test genes with the *lowest* scores:

In [22]:
# GSA: bottom 10% genes (lowest CV)
gmt_bottom = scores_to_gmt(
    scores=specificity,
    out_file="magma_gsa/specificity_bottom10.gmt",
    top_frac=0.10,
    ascending=True,  # select lowest-scoring genes
)

# GPA: negate scores so low-score genes get high covariate values
covar_neg = scores_to_covar(
    scores=specificity,
    out_file="magma_gpa/specificity_negated.covar",
    negate=True,
)

[2026-11-15 19:06:05,949] INFO:cellink.tl.external._ldsc2magma: scores_to_gmt: wrote 31 gene sets (bottom 10%, 0 skipped) → magma_gsa/specificity_bottom10.gmt
[2026-11-15 19:06:06,524] INFO:cellink.tl.external._ldsc2magma: scores_to_covar: wrote 15074 genes × 31 cell types → magma_gpa/specificity_negated.covar


### Passing extra MAGMA flags

Any MAGMA option not explicitly supported can be forwarded via `**kwargs`. Each `key=value` pair becomes `--key value` in the MAGMA call:

In [23]:
# Competitive gene-set test (MAGMA's "multi" model)
run_magma_gsa(
    gene_results=GENES_RAW,
    set_annot=str(gmt_file),
    out_prefix="magma_gsa/cad_competitive",
    magma_bin=MAGMA_BIN,
    model="multi",  # forwarded as --model multi
    run=False,  # preview only
)

{'command': ['magma_bin/magma',
  '--gene-results',
  'magma_results/cad.genes.raw',
  '--set-annot',
  'magma_gsa/onek1k_specificity_top10.gmt',
  '--out',
  'magma_gsa/cad_competitive',
  '--model',
  'multi']}

---
## Summary

This tutorial demonstrated all new functions added to `cellink.tl.external`:

| Function | Purpose |
|---|---|
| `scores_to_gmt` | Convert score DataFrame → MAGMA GMT (top-N% per cell type) |
| `scores_to_covar` | Convert score DataFrame → MAGMA `.covar` (all genes, continuous) |
| `genesets_dir_to_entrez_gmt` | Convert LDSC `.GeneSet` files → MAGMA GMT |
| `run_magma_annotate` | MAGMA Step I: SNP → gene annotation |
| `run_magma_gene_analysis` | MAGMA Step II: gene-level association analysis |
| `run_magma_gsa` | MAGMA Step III-a: gene-set analysis (binary) |
| `run_magma_gpa` | MAGMA Step III-b: gene property analysis (continuous), with optional univariate mode |

Key properties shared across all functions:

* **`run=False`** returns the command list without executing, useful for HPC job submission.
* **Gene ID handling**: `scores_to_gmt` and `scores_to_covar` accept a `gene_map` argument for symbol → ENSG translation; rows that cannot be mapped to ENSG are dropped.
* **Univariate GPA**: `run_magma_gpa(univariate=True)` tests each cell type independently, avoiding MAGMA's collinearity filter when scores are highly correlated.